# UC5.3

In [1]:
import requests
import json

# 設定你的虛擬機 IP 與 IOTA 節點預設的 Port (14265)
# 因為你們剛剛是用 sv1.alexc.one，我們直接連過去
NODE_URL = "http://sv1.alexc.one:14265"

# 準備要發送給 IOTA 私有鏈的標準請求指令 (取得節點資訊)
headers = {'X-IOTA-API-Version': '1', 'Content-Type': 'application/json'}
command = {"command": "getNodeInfo"}

try:
    print(f"正在嘗試連線至 IOTA 私有鏈節點: {NODE_URL} ...")

    # 向虛擬機發送 POST 請求
    response = requests.post(NODE_URL, data=json.dumps(command), headers=headers, timeout=5)

    if response.status_code == 200:
        result = response.json()
        print("\n🎉 連線成功！！！")
        print(f"節點名稱 (AppName): {result.get('appName')}")
        print(f"目前最新里程碑 (LatestMilestoneIndex): {result.get('latestMilestoneIndex')}")
    else:
        print(f"\n❌ 連線失敗，伺服器回應狀態碼: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"\n❌ 無法連線到虛擬機，錯誤訊息: {e}")
    print("💡 提示：請確認虛擬機的防火牆是否有對外開放 14265 Port。")

正在嘗試連線至 IOTA 私有鏈節點: http://sv1.alexc.one:14265 ...

🎉 連線成功！！！
節點名稱 (AppName): IRI Testnet
目前最新里程碑 (LatestMilestoneIndex): 37574


In [2]:
!pip install phx-filters safe-pysha3 requests six
!pip install pyota --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.4/190.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 7.4 MB/s eta 0:00:00


In [3]:
import sys

# 1. 魔法第一步：催眠 pkg_resources，讓它以為系統裡有安裝 pysha3
try:
    import pkg_resources
    dummy_dist = pkg_resources.Distribution(project_name='pysha3', version='1.0.2')
    pkg_resources.working_set.add(dummy_dist)
except Exception:
    pass

# 2. 魔法第二步：將 safe-pysha3 的實體對應給 pyota 看
try:
    import sha3
    sys.modules['pysha3'] = sys.modules['sha3']
except ModuleNotFoundError:
    !pip install safe-pysha3
    import sha3
    sys.modules['pysha3'] = sys.modules['sha3']

/tmp/ipykernel_1713/2917871318.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [ ]:
from iota import Iota, ProposedTransaction, Address, Tag, TryteString
import json

# 連線到你那台 24 小時不打烊的虛擬機
api = Iota("http://sv1.alexc.one:14265")

# 定義跟剛才一模一樣的專屬 A 屋標籤 (必須剛好 27 個字元)
target_tag = Tag("TEST9HOUSE9A999999999999999")
dummy_address = Address("A" * 81) # 早期 IOTA 規定要有 81 字元的地點

# 模擬一筆家庭用電上鏈日誌
log_payload = {
    "timestamp": "2026-07-06T21:30:00Z",
    "device": "Smart_Meter_01",
    "status": "Normal",
    "msg": "I love TWICE!"
}

# 轉碼並打包交易 (value=0 代表純存資料，不轉帳)
message_trytes = TryteString.from_string(json.dumps(log_payload))
tx = ProposedTransaction(address=dummy_address, message=message_trytes, tag=target_tag, value=0)

print("🚀 正在把第一筆 A 屋稽核日誌打入區塊鏈... (正在計算 PoW)")
try:
    # depth=3, min_weight_magnitude=9 是你架的私有鏈設定值
    result = api.send_transfer(depth=3, transfers=[tx], min_weight_magnitude=9)
    print("\n🎉 上傳成功！！！資料已永久刻在 Tangle 上了！")
    print(f"📦 交易 Hash 憑證: {result['bundle'].tail_transaction.hash}")
except Exception as e:
    print(f"❌ 上傳失敗，錯誤訊息: {e}")

🚀 正在把第一筆 A 屋稽核日誌打入區塊鏈... (正在計算 PoW)

🎉 上傳成功！！！資料已永久刻在 Tangle 上了！
📦 交易 Hash 憑證: LGVYDVPOAAMRBBASBQVQJSXOSBXLPOEAAKAFFUEPOPUIEQMJOHRRUVFKDZIWVD9XGVXBZUITDFTRTM999


In [ ]:
import requests
import json

NODE_URL = "http://sv1.alexc.one:14265"
headers = {'X-IOTA-API-Version': '1', 'Content-Type': 'application/json'}

# 模擬 Admin 選定了「房屋 A (HOUSE9A99999999999999999999)」
# 註：IOTA 早期版本的 Tag 必須是 27 個字元的 A-Z 和 9 組合
TARGET_HOUSE_TAG = "TEST9HOUSE9A999999999999999"

print(f"📊 [Admin 行動] 正在查閱場域專屬稽核日誌...")
print(f"🔍 目標場域 Tag: {TARGET_HOUSE_TAG}")

# 向你架好的節點查詢帶有該房屋 Tag 的所有交易雜湊 (Hashes)
find_command = {
    "command": "findTransactions",
    "tags": [TARGET_HOUSE_TAG]
}

try:
    response = requests.post(NODE_URL, data=json.dumps(find_command), headers=headers, timeout=5)
    if response.status_code == 200:
        hashes = response.json().get("hashes", [])
        print(f"\n✅ [成功隔離] 成功撈出該場域專屬的歷史操作紀錄！")
        print(f"📦 找到的交易筆數: {len(hashes)} 筆")
        print(f"📑 稽核日誌清單 (Transaction Hashes):")
        for h in hashes[:3]: # 只印出前三筆示意
            print(f" - {h}")
    else:
        print("❌ 查詢失敗")
except Exception as e:
    print(f"❌ 連線異常: {e}")

📊 [Admin 行動] 正在查閱場域專屬稽核日誌...
🔍 目標場域 Tag: TEST9HOUSE9A999999999999999

✅ [成功隔離] 成功撈出該場域專屬的歷史操作紀錄！
📦 找到的交易筆數: 1 筆
📑 稽核日誌清單 (Transaction Hashes):
 - LGVYDVPOAAMRBBASBQVQJSXOSBXLPOEAAKAFFUEPOPUIEQMJOHRRUVFKDZIWVD9XGVXBZUITDFTRTM999


In [ ]:
from iota import Iota, ProposedTransaction, Address, Tag, TryteString
import json

# 連線到你那台 24 小時不打烊的虛擬機
api = Iota("http://sv1.alexc.one:14265")

# 定義跟剛才一模一樣的專屬 B 屋標籤 (必須剛好 27 個字元)
target_tag = Tag("TEST9HOUSE9B999999999999999")
dummy_address = Address("B" * 81) # 早期 IOTA 規定要有 81 字元的地點

# 模擬一筆家庭用電上鏈日誌
log_payload = {
    "timestamp": "2026-07-06T21:40:00Z",
    "device": "Smart_Meter_01",
    "status": "Normal",
    "msg": "I love TWICE too!"
}

# 轉碼並打包交易 (value=0 代表純存資料，不轉帳)
message_trytes = TryteString.from_string(json.dumps(log_payload))
tx = ProposedTransaction(address=dummy_address, message=message_trytes, tag=target_tag, value=0)

print("🚀 正在把第一筆 B 屋稽核日誌打入區塊鏈... (正在計算 PoW)")
try:
    # depth=3, min_weight_magnitude=9 是你架的私有鏈設定值
    result = api.send_transfer(depth=3, transfers=[tx], min_weight_magnitude=9)
    print("\n🎉 上傳成功！！！資料已永久刻在 Tangle 上了！")
    print(f"📦 交易 Hash 憑證: {result['bundle'].tail_transaction.hash}")
except Exception as e:
    print(f"❌ 上傳失敗，錯誤訊息: {e}")

🚀 正在把第一筆 B 屋稽核日誌打入區塊鏈... (正在計算 PoW)

🎉 上傳成功！！！資料已永久刻在 Tangle 上了！
📦 交易 Hash 憑證: JBQZGCERECK9UIRWDJVAYIC9FJABTKW9NDKACRXTKLZTOJJ9WJZJYDEWLACJFGTXEASNSINYANPNV9999


In [ ]:
import requests
import json

NODE_URL = "http://sv1.alexc.one:14265"
headers = {'X-IOTA-API-Version': '1', 'Content-Type': 'application/json'}

# 註：IOTA 早期版本的 Tag 必須是 27 個字元的 A-Z 和 9 組合
TARGET_HOUSE_TAG = "TEST9HOUSE9B999999999999999"

print(f"📊 [Admin 行動] 正在查閱場域專屬稽核日誌...")
print(f"🔍 目標場域 Tag: {TARGET_HOUSE_TAG}")

# 向你架好的節點查詢帶有該房屋 Tag 的所有交易雜湊 (Hashes)
find_command = {
    "command": "findTransactions",
    "tags": [TARGET_HOUSE_TAG]
}

try:
    response = requests.post(NODE_URL, data=json.dumps(find_command), headers=headers, timeout=5)
    if response.status_code == 200:
        hashes = response.json().get("hashes", [])
        print(f"\n✅ [成功隔離] 成功撈出該場域專屬的歷史操作紀錄！")
        print(f"📦 找到的交易筆數: {len(hashes)} 筆")
        print(f"📑 稽核日誌清單 (Transaction Hashes):")
        for h in hashes[:3]: # 只印出前三筆示意
            print(f" - {h}")
    else:
        print("❌ 查詢失敗")
except Exception as e:
    print(f"❌ 連線異常: {e}")

📊 [Admin 行動] 正在查閱場域專屬稽核日誌...
🔍 目標場域 Tag: TEST9HOUSE9B999999999999999

✅ [成功隔離] 成功撈出該場域專屬的歷史操作紀錄！
📦 找到的交易筆數: 1 筆
📑 稽核日誌清單 (Transaction Hashes):
 - JBQZGCERECK9UIRWDJVAYIC9FJABTKW9NDKACRXTKLZTOJJ9WJZJYDEWLACJFGTXEASNSINYANPNV9999


# UC5.4

調閱 House A 資料給 SP

In [ ]:
import requests
import json
from iota import TryteString  # 用來將區塊鏈的 Trytes 轉回人類文字

NODE_URL = "http://sv1.alexc.one:14265"
headers = {'X-IOTA-API-Version': '1', 'Content-Type': 'application/json'}

# 1. 指定要查詢的場域 (House A)
TARGET_HOUSE_TAG = "TEST9HOUSE9A999999999999999"

print(f"🕵️‍♂️ [SP 廠商行動] 正在持屋主憑證調閱去識別化日誌...")
print(f"🔍 目標場域 Tag: {TARGET_HOUSE_TAG}\n")

# -------------------------------------------------------------
# 步驟 A: 撈取該 Tag 的交易 Hashes (你原本寫好的部分)
# -------------------------------------------------------------
find_command = {
    "command": "findTransactions",
    "tags": [TARGET_HOUSE_TAG]
}

try:
    response = requests.post(NODE_URL, data=json.dumps(find_command), headers=headers, timeout=5)
    if response.status_code == 200:
        hashes = response.json().get("hashes", [])
        print(f"✅ 成功撈出該場域歷史交易，共 {len(hashes)} 筆。開始進行去識別化解碼...\n")

        if not hashes:
            print("📭 目前鏈上沒有該家庭的資料。")
            sys.exit()

        # -------------------------------------------------------------
        # 步驟 B: 拿 Hashes 去跟節點要真正的交易內文 (getTrytes)
        # -------------------------------------------------------------
        # 為了解說方便，我們拿最新的一筆交易 (hashes[-1]) 來做展示
        target_hash = hashes[-1]

        trytes_command = {
            "command": "getTrytes",
            "hashes": [target_hash]
        }

        trytes_res = requests.post(NODE_URL, data=json.dumps(trytes_command), headers=headers, timeout=5)
        if trytes_res.status_code == 200:
            raw_trytes = trytes_res.json().get("trytes", [])[0]

            # 早期 IOTA 交易的前 2187 個字元是 signatureMessageFragment (存放資料的地方)
            message_fragment = raw_trytes[:2187]

            # 將 Trytes 轉回人類看得懂的字串，並去掉末尾填補的 9
            try:
                decoded_string = str(TryteString(message_fragment.encode()).decode())
                # 找到 JSON 結尾的大括號，過濾掉後面無意義的 9999...
                end_index = decoded_string.rfind('}')
                if end_index != -1:
                    decoded_string = decoded_string[:end_index+1]

                # 還原成原始的 JSON 物件
                blockchain_json = json.loads(decoded_string)

                print("==================================================")
                print("🔒 [IOTA 鏈上不可篡改的原始數據] (僅屋主或 Admin 可見):")
                print(json.dumps(blockchain_json, indent=2, ensure_ascii=False))
                print("==================================================")

                # -------------------------------------------------------------
                # 步驟 C: 核心隱私遮蔽演算法 (De-identification) -> 餵給 SP 廠商
                # -------------------------------------------------------------
                def de_identify_for_sp(raw_data):
                    sp_data = raw_data.copy()

                    # 1. 個資敏感欄位強行遮蔽 (假設未來擴充了隱私欄位如使用者姓名)
                    if "user_name" in sp_data:
                        sp_data["user_name"] = sp_data["user_name"][0] + "*" + sp_data["user_name"][-1]

                    # 2. 時間戳記模糊化：把精確的秒數作息抹除，只留到「小時」，保護屋主隱私
                    # 原始格式: 2026-07-06T21:30:00Z -> 變成 2026-07-06T21:00:00Z
                    if "timestamp" in sp_data:
                        sp_data["timestamp"] = sp_data["timestamp"][:14] + "00:00Z"

                    # 3. 敏感的訊息備註 (例如你打的 I love TWICE!) 進行遮蔽
                    if "msg" in sp_data:
                        sp_data["msg"] = "****** (屋主隱私訊息已遮蔽)"

                    return sp_data

                # 執行去識別化
                sp_final_view = de_identify_for_sp(blockchain_json)

                print("\n🛡️ [UC5.4 廠商調閱畫面] (經過去識別化存證日誌):")
                print(json.dumps(sp_final_view, indent=2, ensure_ascii=False))
                print("==================================================")

            except Exception as e:
                print(f"❌ 解析區塊鏈文字失敗: {e}。可能該筆交易內文不是標準 JSON 格式。")

    else:
        print("❌ 查詢失敗")
except Exception as e:
    print(f"❌ 連線異常: {e}")

🕵️‍♂️ [SP 廠商行動] 正在持屋主憑證調閱去識別化日誌...
🔍 目標場域 Tag: TEST9HOUSE9A999999999999999

✅ 成功撈出該場域歷史交易，共 1 筆。開始進行去識別化解碼...

🔒 [IOTA 鏈上不可篡改的原始數據] (僅屋主或 Admin 可見):
{
  "timestamp": "2026-07-06T21:30:00Z",
  "device": "Smart_Meter_01",
  "status": "Normal",
  "msg": "I love TWICE!"
}

🛡️ [UC5.4 廠商調閱畫面] (經過去識別化存證日誌):
{
  "timestamp": "2026-07-06T21:00:00Z",
  "device": "Smart_Meter_01",
  "status": "Normal",
  "msg": "****** (屋主隱私訊息已遮蔽)"
}
